# 🧹 Cell 7 — Data Cleaning
تنظيف بيانات Property Finder Egypt وتصدير ملف Excel منظم

**يتطلب:** `propertyfinder_properties.csv` في نفس المجلد

**ينتج:** `propertyfinder_cleaned.xlsx` بـ 4 شيتات

In [ ]:
# ════════════════════════════════════════════════════════════════════
#  Cell 7 — Data Cleaning
#  تنظيف البيانات وتصدير ملف Excel منظم
# ════════════════════════════════════════════════════════════════════

import re
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# ── Load ─────────────────────────────────────────────────────────
df = pd.read_csv("propertyfinder_properties.csv")
print(f"📂 Loaded: {df.shape[0]} rows × {df.shape[1]} cols")

# ── Arabic → Western digits ───────────────────────────────────────
AR_DIGITS = str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789")
def ar_to_en(t):
    return t.translate(AR_DIGITS) if isinstance(t, str) else t

# ── Price ─────────────────────────────────────────────────────────
def extract_price_period(s):
    if not isinstance(s, str): return None
    if "شهرياً" in s or "شهري" in s: return "شهري"
    if "سنوياً" in s or "سنوي" in s: return "سنوي"
    return None

def clean_price(s):
    if pd.isna(s): return None
    s = ar_to_en(str(s))
    s = re.sub(r"[٬,]", "", s)
    s = re.sub(r"جنيه.*", "", s).strip()
    try: return int(float(s))
    except: return None

df["price_period"] = df["price"].apply(extract_price_period)
df["price_egp"]    = df["price"].apply(clean_price)

# ── Area ─────────────────────────────────────────────────────────
def clean_area(s):
    if pd.isna(s): return None
    s = ar_to_en(str(s))
    d = re.sub(r"[^\d.]", "", s)
    try: return float(d)
    except: return None

df["area_sqm"] = df["area"].apply(clean_area)

# ── Bedrooms / Bathrooms ─────────────────────────────────────────
def clean_rooms(s):
    if pd.isna(s): return None
    s = str(s).strip().replace("+", "")
    if s == "استوديو": return "استوديو"
    try: return int(s)
    except: return s

df["bedrooms_clean"]  = df["bedrooms"].apply(clean_rooms)
df["bathrooms_clean"] = df["bathrooms"].apply(clean_rooms)

# ── Location split ────────────────────────────────────────────────
def split_location(loc):
    if pd.isna(loc): return None, None, None, None
    parts = [p.strip() for p in str(loc).split(",")]
    n = len(parts)
    return (parts[0] if n >= 2 else None,
            parts[-3] if n >= 3 else None,
            parts[-2] if n >= 2 else None,
            parts[-1])

locs = df["location"].apply(split_location)
df["compound"]    = [x[0] for x in locs]
df["district"]    = [x[1] for x in locs]
df["city"]        = [x[2] for x in locs]
df["governorate"] = [x[3] for x in locs]

# ── Image count ───────────────────────────────────────────────────
df["image_count"] = df["image_urls"].apply(
    lambda x: len(str(x).split("|")) if pd.notna(x) else 0)

# ── Dedup & filter ────────────────────────────────────────────────
before = len(df)
df = df.drop_duplicates(subset=["listing_url"], keep="first")
df = df[df["price_egp"].notna()].reset_index(drop=True)
print(f"🧹 Removed {before - len(df)} duplicate/invalid rows → {len(df)} clean rows")

# ── Final column order ────────────────────────────────────────────
COLS = {
    "title":           "العنوان",
    "listing_type":    "نوع الإعلان",
    "property_type":   "نوع العقار",
    "price_egp":       "السعر (جنيه)",
    "price_period":    "فترة السعر",
    "area_sqm":        "المساحة (م²)",
    "bedrooms_clean":  "غرف النوم",
    "bathrooms_clean": "الحمامات",
    "compound":        "الكمبوند / المشروع",
    "district":        "الحي / المنطقة",
    "city":            "المدينة",
    "governorate":     "المحافظة",
    "agency_name":     "اسم الوكالة",
    "publish_date":    "تاريخ النشر",
    "publish_time":    "وقت النشر",
    "image_count":     "عدد الصور",
    "listing_url":     "رابط الإعلان",
    "agency_logo":     "لوجو الوكالة",
    "image_urls":      "روابط الصور",
    "source":          "المصدر",
}
clean_df = df[[c for c in COLS]].copy()
clean_df.columns = list(COLS.values())

# ════════════════════════════════════════════════════════════════════
#  BUILD EXCEL
# ════════════════════════════════════════════════════════════════════
HDR_FILL  = PatternFill("solid", fgColor="1F3864")
HDR_FONT  = Font(name="Arial", bold=True, color="FFFFFF", size=11)
SECT_FILL = PatternFill("solid", fgColor="2E74B5")
SECT_FONT = Font(name="Arial", bold=True, color="FFFFFF", size=11)
DATA_FONT = Font(name="Arial", size=10)
CENTER    = Alignment(horizontal="center", vertical="center", wrap_text=True)
R_ALIGN   = Alignment(horizontal="right",  vertical="center", wrap_text=True)
thin      = Side(style="thin", color="D9D9D9")
BORDER    = Border(left=thin, right=thin, top=thin, bottom=thin)
NUM_FMT   = "#,##0"

COL_W = {
    "العنوان": 45, "نوع الإعلان": 13, "نوع العقار": 14,
    "السعر (جنيه)": 16, "فترة السعر": 12, "المساحة (م²)": 13,
    "غرف النوم": 11, "الحمامات": 11, "الكمبوند / المشروع": 25,
    "الحي / المنطقة": 20, "المدينة": 20, "المحافظة": 14,
    "اسم الوكالة": 22, "تاريخ النشر": 14, "وقت النشر": 12,
    "عدد الصور": 11, "رابط الإعلان": 55, "لوجو الوكالة": 55,
    "روابط الصور": 40, "المصدر": 20,
}

def write_sheet(ws, data, headers):
    ws.sheet_view.rightToLeft = True
    for ci, h in enumerate(headers, 1):
        c = ws.cell(row=1, column=ci, value=h)
        c.fill = HDR_FILL; c.font = HDR_FONT
        c.alignment = CENTER; c.border = BORDER
    ws.row_dimensions[1].height = 32

    for ri, row_data in enumerate(data.itertuples(index=False), 2):
        for ci, val in enumerate(row_data, 1):
            c = ws.cell(row=ri, column=ci, value=val if pd.notna(val) else None)
            c.font = DATA_FONT; c.border = BORDER; c.alignment = R_ALIGN
            h = headers[ci - 1]
            if h in ("السعر (جنيه)", "المساحة (م²)", "عدد الصور"):
                c.number_format = NUM_FMT
                c.alignment = Alignment(horizontal="center", vertical="center")
            elif h in ("غرف النوم", "الحمامات"):
                c.alignment = Alignment(horizontal="center", vertical="center")

    for ci, h in enumerate(headers, 1):
        ws.column_dimensions[get_column_letter(ci)].width = COL_W.get(h, 15)
    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions

wb   = Workbook()
sale = clean_df[clean_df["نوع الإعلان"] == "Sale"]
rent = clean_df[clean_df["نوع الإعلان"] == "Rent"]
headers = list(clean_df.columns)

# Sheet 1 — all data
ws1 = wb.active; ws1.title = "البيانات المنظفة"
write_sheet(ws1, clean_df, headers)

# Sheet 2 — Summary Dashboard
ws2 = wb.create_sheet("ملخص")
ws2.sheet_view.rightToLeft = True
ws2.merge_cells("A1:F1")
t = ws2["A1"]
t.value = "📊 ملخص بيانات Property Finder Egypt"
t.font = Font(name="Arial", bold=True, size=16, color="FFFFFF")
t.fill = PatternFill("solid", fgColor="1F3864")
t.alignment = Alignment(horizontal="center", vertical="center")
ws2.row_dimensions[1].height = 42

def sec_hdr(ws, row, col, text, span=2):
    if span > 1:
        ws.merge_cells(start_row=row, start_column=col, end_row=row, end_column=col+span-1)
    c = ws.cell(row=row, column=col, value=text)
    c.fill = SECT_FILL; c.font = SECT_FONT
    c.alignment = CENTER; c.border = BORDER
    ws.row_dimensions[row].height = 24

def kv(ws, row, col, label, value, num=False):
    lc = ws.cell(row=row, column=col, value=label)
    vc = ws.cell(row=row, column=col+1, value=value)
    lc.font = Font(name="Arial", bold=True, size=10)
    lc.fill = PatternFill("solid", fgColor="EBF3FB")
    lc.border = BORDER; lc.alignment = R_ALIGN
    vc.font = DATA_FONT; vc.border = BORDER
    vc.alignment = Alignment(horizontal="center", vertical="center")
    if num: vc.number_format = NUM_FMT
    ws.row_dimensions[row].height = 20

sec_hdr(ws2,  3, 1, "📋 إجمالي البيانات")
kv(ws2,  4, 1, "إجمالي الإعلانات",  len(clean_df), num=True)
kv(ws2,  5, 1, "إعلانات البيع",     len(sale),     num=True)
kv(ws2,  6, 1, "إعلانات الإيجار",   len(rent),     num=True)
kv(ws2,  7, 1, "وكالات مختلفة",     clean_df["اسم الوكالة"].nunique(), num=True)
kv(ws2,  8, 1, "مناطق مختلفة",      clean_df["الحي / المنطقة"].nunique(), num=True)
kv(ws2,  9, 1, "محافظات مختلفة",    clean_df["المحافظة"].nunique(), num=True)

sec_hdr(ws2, 3, 4, "💰 أسعار البيع (جنيه)")
kv(ws2,  4, 4, "متوسط السعر", int(sale["السعر (جنيه)"].mean()) if len(sale) else 0, num=True)
kv(ws2,  5, 4, "أقل سعر",    int(sale["السعر (جنيه)"].min())  if len(sale) else 0, num=True)
kv(ws2,  6, 4, "أعلى سعر",   int(sale["السعر (جنيه)"].max())  if len(sale) else 0, num=True)
kv(ws2,  7, 4, "الوسيط",     int(sale["السعر (جنيه)"].median()) if len(sale) else 0, num=True)

sec_hdr(ws2,  9, 4, "🏠 أسعار الإيجار (جنيه/شهر)")
kv(ws2, 10, 4, "متوسط السعر", int(rent["السعر (جنيه)"].mean()) if len(rent) else 0, num=True)
kv(ws2, 11, 4, "أقل سعر",    int(rent["السعر (جنيه)"].min())  if len(rent) else 0, num=True)
kv(ws2, 12, 4, "أعلى سعر",   int(rent["السعر (جنيه)"].max())  if len(rent) else 0, num=True)

sec_hdr(ws2, 14, 1, "🏗️ أنواع العقارات")
for i, (pt, cnt) in enumerate(clean_df["نوع العقار"].value_counts().head(8).items()):
    kv(ws2, 15+i, 1, pt, cnt, num=True)

sec_hdr(ws2, 14, 4, "📍 أعلى المحافظات")
for i, (gov, cnt) in enumerate(clean_df["المحافظة"].value_counts().head(8).items()):
    kv(ws2, 15+i, 4, gov, cnt, num=True)

sec_hdr(ws2, 25, 1, "✅ جودة البيانات", span=4)
qa = [
    ("إجمالي الصفوف بعد التنظيف", len(clean_df)),
    ("بدون اسم وكالة",            int(clean_df["اسم الوكالة"].isna().sum())),
    ("بدون غرف نوم",              int(clean_df["غرف النوم"].isna().sum())),
    ("بدون صور",                  int((clean_df["عدد الصور"] == 0).sum())),
    ("تكرار محذوف",               before - len(df)),
]
for i, (lbl, val) in enumerate(qa):
    kv(ws2, 26+i, 1, lbl, val, num=True)

for col, w in zip(["A","B","C","D","E","F"], [26,18,4,26,18,4]):
    ws2.column_dimensions[col].width = w

# Sheet 3 — Sale only
ws3 = wb.create_sheet("إعلانات البيع")
sale_h = [c for c in headers if c != "فترة السعر"]
write_sheet(ws3, sale[[c for c in sale.columns if c != "فترة السعر"]], sale_h)

# Sheet 4 — Rent only
ws4 = wb.create_sheet("إعلانات الإيجار")
write_sheet(ws4, rent, headers)

wb.save("propertyfinder_cleaned.xlsx")
print(f"\n✅ Saved: propertyfinder_cleaned.xlsx")
print(f"   📄 البيانات المنظفة : {len(clean_df)} إعلان")
print(f"   📊 ملخص             : dashboard")
print(f"   🏠 إعلانات البيع    : {len(sale)} إعلان")
print(f"   🏢 إعلانات الإيجار  : {len(rent)} إعلان")
clean_df.head()
